## AQI classification with new SNN encoding


## Preparing data
The dataset used is hourly air quality data (2015 - 2020) from various measuring stations across India: https://www.kaggle.com/rohanrao/air-quality-data-in-india

We'll use one city (Thiruvananthapuram in Kerala) that has two stations and compare it with the actual AQI values present in the data at station, city, hour and day level to confirm the calculations are correct.


In [56]:
## importing packages
import numpy as np
import pandas as pd

In [57]:
## defining constants
PATH_STATION_HOUR = "./data/station_hour.csv"
PATH_STATIONS = "./data/stations.csv"

In [58]:
## importing data and subsetting the station
df = pd.read_csv(PATH_STATION_HOUR, parse_dates=["Datetime"], low_memory=False)
stations = pd.read_csv(PATH_STATIONS)

df = df.merge(stations, on="StationId")

df.sort_values(["StationId", "Datetime"], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Loaded {len(df):,} rows, {df.StationId.nunique()} stations")

Loaded 2,589,083 rows, 110 stations


# SNN Part

SNN-focused pipeline:

- foundation 24-hour window contract
- breakpoint-aware Gaussian population encoding (absolute channels)
- deterministic latency spikes (no random rate coding)
- population-path SNN training and evaluation


In [59]:
## SNN imports
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix

import snntorch as snn


### Foundation data contract
Build a time-safe split before windowing, then define one 24-hour station sample contract with: raw pollutant values [24, 7], missing-mask [24, 7], hour/month metadata per timestep, and label.

This notebook is population-path focused and uses the new deterministic Gaussian absolute encoding pipeline.

In [60]:
FEATURES = ["PM2.5", "PM10", "SO2", "NOx", "NH3", "CO", "O3"]
BUCKET_TO_INT = {
    "Good": 0,
    "Satisfactory": 1,
    "Moderate": 2,
    "Poor": 3,
    "Very Poor": 4,
    "Severe": 5,
}
INT_TO_BUCKET = {v: k for k, v in BUCKET_TO_INT.items()}
WINDOW_SIZE = 24

# DEV mode to run fast iteration
DEV_MODE = False

# Selection toggle for training samples:
# true - all data (like it is right now);
# false - only rows that are incomplete (taking into account that some stations have missing pollutant readings in all
# rows, so if row is incomplete but these pollutants are missing in all other rows of this station, then it's also complete, since the stastion doesn't have sensors for this
# pollutant)
ALL_DATA_SELECTION = False

DELTA_STEPS_FULL = 8
DELTA_STEPS_DEV = 4
DELTA_THRESHOLD_STD = 0.25
DELTA_SATURATION_STD = 3.0
DELTA_IQR_FLOOR = 1e-6
EPOCHS_DELTA_FULL = 10
EPOCHS_DELTA_DEV = 3
TRAIN_BATCH_SIZE_FULL = 256
TRAIN_BATCH_SIZE_DEV = 512
EVAL_BATCH_SIZE_FULL = 512
EVAL_BATCH_SIZE_DEV = 512
MAX_TRAIN_BATCHES_DEV = 80
MAX_EVAL_BATCHES_DEV = 30
NUM_WORKERS = min(4, max(1, os.cpu_count() or 1))

DELTA_STEPS_ACTIVE = DELTA_STEPS_DEV if DEV_MODE else DELTA_STEPS_FULL
if DELTA_SATURATION_STD <= DELTA_THRESHOLD_STD:
    raise ValueError("DELTA_SATURATION_STD must be > DELTA_THRESHOLD_STD")

# Temporary compatibility for downstream cells copied from the population notebook.
POPULATION_STEPS_ACTIVE = DELTA_STEPS_ACTIVE
EPOCHS_DELTA_ACTIVE = EPOCHS_DELTA_DEV if DEV_MODE else EPOCHS_DELTA_FULL
TRAIN_BATCH_SIZE_ACTIVE = TRAIN_BATCH_SIZE_DEV if DEV_MODE else TRAIN_BATCH_SIZE_FULL
EVAL_BATCH_SIZE_ACTIVE = EVAL_BATCH_SIZE_DEV if DEV_MODE else EVAL_BATCH_SIZE_FULL
MAX_TRAIN_BATCHES_ACTIVE = MAX_TRAIN_BATCHES_DEV if DEV_MODE else None
MAX_EVAL_BATCHES_ACTIVE = MAX_EVAL_BATCHES_DEV if DEV_MODE else None

# Use existing AQI bucket from dataset as target column for classification.
target_col = "AQI_Bucket"

base_df = (
    df.loc[:, ["StationId", "Datetime", *FEATURES, target_col]]
    .rename(columns={target_col: "target_bucket"})
    .copy()
)
base_df["Datetime"] = pd.to_datetime(base_df["Datetime"], errors="coerce")

valid_rows = (
    base_df["StationId"].notna()
    & base_df["Datetime"].notna()
    & base_df["target_bucket"].notna()
    & (base_df["target_bucket"].astype(str) != "0")
)
base_df = base_df.loc[valid_rows].copy()

base_df["label"] = base_df["target_bucket"].map(BUCKET_TO_INT)
base_df = (
    base_df.loc[base_df["label"].notna()]
    .assign(label=lambda d: d["label"].astype(int))
    .sort_values(["Datetime", "StationId"])
    .reset_index(drop=True)
)

print(f"Rows with valid AQI bucket: {len(base_df):,}")
print(base_df["target_bucket"].value_counts())
print(
    f"Runtime profile -> DEV_MODE={DEV_MODE}, steps={DELTA_STEPS_ACTIVE}, threshold={DELTA_THRESHOLD_STD}, saturation={DELTA_SATURATION_STD}, epochs={EPOCHS_DELTA_ACTIVE}, train_bs={TRAIN_BATCH_SIZE_ACTIVE}, eval_bs={EVAL_BATCH_SIZE_ACTIVE}, train_batch_cap={MAX_TRAIN_BATCHES_ACTIVE}, workers={NUM_WORKERS}"
)
print(f"Datetime span: {base_df['Datetime'].min()} -> {base_df['Datetime'].max()}")

Rows with valid AQI bucket: 2,018,893
target_bucket
Moderate        675008
Satisfactory    530164
Very Poor       301150
Poor            239990
Good            152113
Severe          120468
Name: count, dtype: int64
Runtime profile -> DEV_MODE=False, steps=8, threshold=0.25, saturation=3.0, epochs=10, train_bs=256, eval_bs=512, train_batch_cap=None, workers=4
Datetime span: 2015-01-01 16:00:00 -> 2020-07-01 00:00:00


## Foundation: time-safe split before windowing + sample contract

This cell also supports two selection modes for experiments:
- all windows (default)
- only incomplete windows after excluding station-level structural missing pollutants (likely no sensor).

In [61]:
## Split by time BEFORE windowing (train/val/test = 80/10/10 by timestamp)
unique_times = np.sort(base_df["Datetime"].unique())
train_cut_idx = int(len(unique_times) * 0.8)
val_cut_idx = int(len(unique_times) * 0.9)
train_cutoff = unique_times[max(train_cut_idx - 1, 0)]
val_cutoff = unique_times[max(val_cut_idx - 1, 0)]

split_train_df = base_df[base_df["Datetime"] <= train_cutoff].copy()
split_val_df = base_df[
    (base_df["Datetime"] > train_cutoff) & (base_df["Datetime"] <= val_cutoff)
].copy()
split_test_df = base_df[base_df["Datetime"] > val_cutoff].copy()

# Station-level structural missingness (all rows missing for pollutant => likely no sensor).
station_observed_counts = base_df.groupby("StationId")[FEATURES].count()
station_structural_missing = station_observed_counts.eq(0)
station_structural_missing_map = {
    station_id: row.to_numpy(dtype=bool)
    for station_id, row in station_structural_missing.iterrows()
}


class AQIWindowDataset(torch.utils.data.Dataset):
    """Lazy 24-hour window dataset with explicit missing mask + time metadata."""

    def __init__(
        self,
        split_df,
        feature_cols,
        window_size=24,
        station_structural_missing_map=None,
        all_data_selection=True,
    ):
        self.feature_cols = feature_cols
        self.window_size = window_size
        self.station_data = {}
        self.window_index = []
        self.dropped_gap_windows = 0
        self.dropped_complete_windows = 0
        self.total_contiguous_endpoints = 0
        self.station_structural_missing_map = station_structural_missing_map or {}

        self.all_data_selection = bool(all_data_selection)

        for station_id, station_df in split_df.groupby("StationId", sort=False):
            station_df = station_df.sort_values("Datetime").reset_index(drop=True)
            if len(station_df) < window_size:
                continue

            features_np = station_df[feature_cols].to_numpy(dtype=np.float32)

            # Endpoint is valid only if the previous (window_size - 1) gaps are exactly 1 hour.
            diffs = station_df["Datetime"].diff().dt.total_seconds()
            contiguous = diffs.eq(3600)
            valid_end = (
                contiguous.rolling(window=window_size - 1, min_periods=window_size - 1)
                .sum()
                .eq(window_size - 1)
            )
            valid_end = valid_end.fillna(False)
            contiguous_end_indices = np.flatnonzero(valid_end.to_numpy())
            self.total_contiguous_endpoints += len(contiguous_end_indices)
            if len(contiguous_end_indices) == 0:
                continue

            selected_end_indices = contiguous_end_indices
            if not self.all_data_selection:
                structural_missing = self.station_structural_missing_map.get(station_id)
                if structural_missing is None:
                    structural_missing = np.zeros(len(feature_cols), dtype=bool)

                effective_missing = np.isnan(features_np) & (
                    ~structural_missing[None, :]
                )
                row_is_incomplete = effective_missing.any(axis=1)
                selected_end_indices = contiguous_end_indices[
                    row_is_incomplete[contiguous_end_indices]
                ]

            if len(selected_end_indices) == 0:
                continue

            self.station_data[station_id] = {
                "features": features_np,
                "hour": station_df["Datetime"].dt.hour.to_numpy(dtype=np.int16),
                "month": station_df["Datetime"].dt.month.to_numpy(dtype=np.int16),
                "label": station_df["label"].to_numpy(dtype=np.int64),
                "datetime": station_df["Datetime"].to_numpy(),
            }

            for end_idx in selected_end_indices:
                self.window_index.append((station_id, int(end_idx)))

            possible_endpoints = max(len(station_df) - window_size + 1, 0)
            self.dropped_gap_windows += possible_endpoints - len(contiguous_end_indices)
            self.dropped_complete_windows += len(contiguous_end_indices) - len(
                selected_end_indices
            )

    def __len__(self):
        return len(self.window_index)

    def __getitem__(self, idx):
        station_id, end_idx = self.window_index[idx]
        station = self.station_data[station_id]
        start_idx = end_idx - self.window_size + 1

        raw = station["features"][start_idx : end_idx + 1]
        missing_mask = np.isnan(raw).astype(np.uint8)
        hours = station["hour"][start_idx : end_idx + 1]
        months = station["month"][start_idx : end_idx + 1]
        label = int(station["label"][end_idx])

        return {
            "raw": raw,  # [24, 7]
            "missing_mask": missing_mask,  # [24, 7]
            "hour": hours,  # [24]
            "month": months,  # [24]
            "label": label,
            "station_id": station_id,
            "end_datetime": station["datetime"][end_idx],
        }


foundation_train = AQIWindowDataset(
    split_train_df,
    FEATURES,
    window_size=WINDOW_SIZE,
    station_structural_missing_map=station_structural_missing_map,
    all_data_selection=ALL_DATA_SELECTION,
)
foundation_val = AQIWindowDataset(
    split_val_df,
    FEATURES,
    window_size=WINDOW_SIZE,
    station_structural_missing_map=station_structural_missing_map,
    all_data_selection=ALL_DATA_SELECTION,
)
foundation_test = AQIWindowDataset(
    split_test_df,
    FEATURES,
    window_size=WINDOW_SIZE,
    station_structural_missing_map=station_structural_missing_map,
    all_data_selection=ALL_DATA_SELECTION,
)

print(
    f"Time cutoffs -> train: <= {train_cutoff}, val: <= {val_cutoff}, test: > {val_cutoff}"
)
print(f"ALL_DATA_SELECTION: {ALL_DATA_SELECTION}")
print(
    f"Window samples -> train: {len(foundation_train):,}, val: {len(foundation_val):,}, test: {len(foundation_test):,}"
)
print(
    "Contiguous endpoints before completeness filter -> "
    f"train: {foundation_train.total_contiguous_endpoints:,}, "
    f"val: {foundation_val.total_contiguous_endpoints:,}, "
    f"test: {foundation_test.total_contiguous_endpoints:,}"
)
print(
    "Contiguous endpoints total (all splits): "
    f"{(foundation_train.total_contiguous_endpoints + foundation_val.total_contiguous_endpoints + foundation_test.total_contiguous_endpoints):,}"
)
print(
    "Dropped windows due to gaps -> "
    f"train: {foundation_train.dropped_gap_windows:,}, "
    f"val: {foundation_val.dropped_gap_windows:,}, "
    f"test: {foundation_test.dropped_gap_windows:,}"
)
if not ALL_DATA_SELECTION:
    print(
        "Dropped windows by completeness filter -> "
        f"train: {foundation_train.dropped_complete_windows:,}, "
        f"val: {foundation_val.dropped_complete_windows:,}, "
        f"test: {foundation_test.dropped_complete_windows:,}"
    )
print("Stations with structural-missing channels per pollutant:")
print(station_structural_missing.sum().to_dict())

if len(foundation_train) > 0:
    sample = foundation_train[0]
    print("Sample contract shapes:")
    print(
        f"raw={sample['raw'].shape}, missing_mask={sample['missing_mask'].shape}, hour={sample['hour'].shape}, month={sample['month'].shape}, label={sample['label']}"
    )

Time cutoffs -> train: <= 2019-05-26T12:00:00.000000, val: <= 2019-12-13T06:00:00.000000, test: > 2019-12-13T06:00:00.000000
ALL_DATA_SELECTION: False
Window samples -> train: 279,205, val: 58,352, test: 89,591
Contiguous endpoints before completeness filter -> train: 1,090,299, val: 395,102, test: 441,346
Contiguous endpoints total (all splits): 1,926,747
Dropped windows due to gaps -> train: 56,115, val: 15,048, test: 14,466
Dropped windows by completeness filter -> train: 811,094, val: 336,750, test: 351,755
Stations with structural-missing channels per pollutant:
{'PM2.5': 2, 'PM10': 17, 'SO2': 9, 'NOx': 2, 'NH3': 26, 'CO': 0, 'O3': 7}
Sample contract shapes:
raw=(24, 7), missing_mask=(24, 7), hour=(24,), month=(24,), label=4


In [62]:
DELTA_THR = 0.05  # basic threshold (tune later)

def encode_delta_on_off(raw_24x7, thr=DELTA_THR):
    # raw_24x7: [24, 7]
    x = np.nan_to_num(raw_24x7, nan=0.0).astype(np.float32)
    deltas = np.diff(x, axis=0, prepend=x[[0]])

    on  = (deltas >  thr).astype(np.float32)
    off = (deltas < -thr).astype(np.float32) 

    spikes_delta = np.concatenate([on, off], axis=1)# [24, 14]
    return spikes_delta

X_delta = []
y = []
meta = []

# for i in range(len(foundation_train)):
#     sample = foundation_train[i]  # dict with raw, label, station_id, ...
#     spikes_delta = encode_delta_on_off(sample["raw"], thr=DELTA_THR)  # [24, 14]

#     X_delta.append(spikes_delta)
#     y.append(sample["label"])
#     meta.append((sample["station_id"], sample["end_datetime"]))

# X_delta = np.stack(X_delta).astype(np.float32)  # [N, 24, 14]
# y = np.asarray(y, dtype=np.int64)               # [N]

# print(X_delta.shape, y.shape)

for i in range(3):
    sample = foundation_train[i]
    print(i, encode_delta_on_off(sample["raw"]).shape, sample["label"])

0 (24, 14) 4
1 (24, 14) 4
2 (24, 14) 4


In [63]:
for feat_idx in range(7):
    all_values = []
    for sample in foundation_train:
        raw = sample['raw']
        pollutant_column = raw[:, feat_idx]  # all 24 hours for this pollutant
        all_values.append(pollutant_column)

    # Flatten into one 1D array
    all_values_flat = np.concatenate(all_values)  # shape (24 * len(train),)

    # Filter to finite values only (drop NaN)
    finite_values = all_values_flat[np.isfinite(all_values_flat)]

    # Compute stats
    min_val = np.min(finite_values)
    max_val = np.max(finite_values)
    median_val = np.median(finite_values)

    print(f"Pollutant {FEATURES[feat_idx]}: min={min_val:.2e}, max={max_val:.2e}, median={median_val:.2e}")

Pollutant PM2.5: min=1.00e-02, max=1.00e+03, median=6.00e+01
Pollutant PM10: min=1.00e-02, max=1.00e+03, median=1.36e+02
Pollutant SO2: min=1.00e-02, max=2.00e+02, median=7.17e+00
Pollutant NOx: min=0.00e+00, max=5.00e+02, median=2.36e+01
Pollutant NH3: min=1.00e-02, max=5.00e+02, median=2.56e+01
Pollutant CO: min=0.00e+00, max=4.99e+02, median=1.07e+00
Pollutant O3: min=1.00e-02, max=9.96e+02, median=2.37e+01


In [64]:
MICRO_STEPS = 8
DELTA_THRESHOLD_STD = 0.25
DELTA_SATURATION_STD = 3.0
DELTA_IQR_FLOOR = 1e-6

print(f"MICRO_STEPS={MICRO_STEPS}")
print(f"DELTA_THRESHOLD_STD={DELTA_THRESHOLD_STD}, DELTA_SATURATION_STD={DELTA_SATURATION_STD}, DELTA_IQR_FLOOR={DELTA_IQR_FLOOR}")


MICRO_STEPS=8
DELTA_THRESHOLD_STD=0.25, DELTA_SATURATION_STD=3.0, DELTA_IQR_FLOOR=1e-06


# Delta ON/OFF Encoder Output Contract

## Output Shape
`[192, 14]` = 24 hours × 8 micro-steps per hour, 7 pollutants × 2 channels (ON/OFF)

## Channel Layout

| Pollutant | ON Channel | OFF Channel |
|-----------|-----------|------------|
| PM2.5 | 0 | 1 |
| PM10 | 2 | 3 |
| SO2 | 4 | 5 |
| NOx | 6 | 7 |
| NH3 | 8 | 9 |
| CO | 10 | 11 |
| O3 | 12 | 13 |

In [65]:
def build_delta_robust_config(train_df, feature_cols, iqr_floor):
    config = {}
    for feat in feature_cols:
        col = train_df[feat].to_numpy(dtype=np.float32)
        finite_values = col[np.isfinite(col)]

        if len(finite_values) == 0:
            print(f"Warning: {feat} — no finite values, using fallback (median=0.0, iqr=1.0).")
            config[feat] = {"raw_median": 0.0, "median": 0.0, "iqr": 1.0}
            continue

        compressed_values = np.log1p(np.clip(finite_values, 0, None))

        raw_median = float(np.median(finite_values))
        median = float(np.median(compressed_values))
        q1 = np.percentile(compressed_values, 25)
        q3 = np.percentile(compressed_values, 75)
        iqr = max(q3 - q1, iqr_floor)

        print(f"{feat}: n={len(finite_values):,}  raw_median={raw_median:.4f}  median={median:.4f}  Q1={q1:.4f}  Q3={q3:.4f}  IQR={iqr:.4f}")
        config[feat] = {"raw_median": raw_median, "median": median, "iqr": iqr}

    return config

robust_config = build_delta_robust_config(split_train_df, FEATURES, iqr_floor=DELTA_IQR_FLOOR)

all_ok = True
for feat, cfg in robust_config.items():
    z = (np.log1p(cfg["raw_median"]) - cfg["median"]) / cfg["iqr"]
    status = "OK" if abs(z) < 0.05 else "FAIL"
    if status == "FAIL":
        all_ok = False
    print(f"  {feat}: z={z:.6f}  [{status}]")
print("All checks passed." if all_ok else "SOME CHECKS FAILED.")

PM2.5: n=1,062,582  raw_median=62.4700  median=4.1506  Q1=3.5893  Q3=4.7585  IQR=1.1692
PM10: n=723,755  raw_median=143.1700  median=4.9710  Q1=4.4397  Q3=5.5152  IQR=1.0755
SO2: n=970,039  raw_median=8.4400  median=2.2450  Q1=1.6658  Q3=2.8344  IQR=1.1686
NOx: n=1,032,202  raw_median=26.7000  median=3.3214  Q1=2.7000  Q3=3.9578  IQR=1.2577
NH3: n=659,650  raw_median=25.8400  median=3.2899  Q1=2.6933  Q3=3.7718  IQR=1.0786
CO: n=1,012,813  raw_median=0.9400  median=0.6627  Q1=0.4318  Q3=0.9439  IQR=0.5121
O3: n=967,327  raw_median=26.1600  median=3.3017  Q1=2.5120  Q3=4.0004  IQR=1.4884
  PM2.5: z=-0.000000  [OK]
  PM10: z=0.000000  [OK]
  SO2: z=-0.000000  [OK]
  NOx: z=0.000000  [OK]
  NH3: z=-0.000000  [OK]
  CO: z=0.000000  [OK]
  O3: z=-0.000000  [OK]
All checks passed.


In [ ]:
def encode_signed_delta_on_off(
    raw,
    missing_mask,
    micro_steps=DELTA_STEPS_ACTIVE,
    threshold_std=DELTA_THRESHOLD_STD,
    saturation_std=DELTA_SATURATION_STD,
    config=None,
    feature_names=None,
):
    raw = raw.astype(np.float32)
    missing_mask = missing_mask.astype(bool)
    n_hours, n_feats = raw.shape

    if config is None:
        config = robust_config
    if feature_names is None:
        feature_names = FEATURES[:n_feats]

    spikes = np.zeros((n_hours * micro_steps, 2 * n_feats), dtype=np.uint8)
    reference = np.full(n_feats, np.nan, dtype=np.float32)
    denom = max(saturation_std - threshold_std, 1e-6)

    for h in range(n_hours):
        for feat_idx in range(n_feats):
            if missing_mask[h, feat_idx] or not np.isfinite(raw[h, feat_idx]):
                continue

            value = float(raw[h, feat_idx])
            feat = feature_names[feat_idx]
            median = config[feat]["median"]
            iqr = config[feat]["iqr"]
            value_log = np.log1p(max(value, 0.0))
            z = (value_log - median) / iqr

            prev = reference[feat_idx]
            reference[feat_idx] = z
            if np.isnan(prev):
                continue

            delta = z - prev
            abs_delta = abs(delta)
            if abs_delta < threshold_std:
                continue

            strength = (min(abs_delta, saturation_std) - threshold_std) / denom
            tau = 1 + int(np.floor((1.0 - strength) * (micro_steps - 1)))
            global_t = h * micro_steps + (tau - 1)  # 4.3

            if delta >= 0:                            # 4.4
                spikes[global_t, 2 * feat_idx] = 1
            else:
                spikes[global_t, 2 * feat_idx + 1] = 1

    return spikes

print("encode_signed_delta_on_off defined")

encode_signed_delta_on_off defined
